## Description:  

### This script aggregates the oberved water quality data by subbasin. Data from all stations within a HydroBasin are merged on daily basis. 

## Input: 
* merged water quality data (0km distance stations merged and assigned new ID) :wq_data_aggregated_0km_dist_new_ID.csv 
* corresponding shapefile containing all unique stations: unique_stations_aggregated_IDs_0km_dist.parquet
* HYDROBASINS : input_data\geodatata\HydroBasins\BasinATLAS_v10.gdb'



In [4]:
import pandas as pd
import numpy as np
import geopandas as gpd
import fiona
import time
import dask_geopandas
from tqdm import tqdm
import pickle
import os

In [5]:
# define the output folder path:

output_folder = '../../output_data/aggregate_stations_by_subbasins'

# Check if the folder exists, if not, create it:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

### read in HYDROBASINs shapefile: 




In [2]:
gdb_path = f'../../geodata/HydroBasins/BasinATLAS_v10.gdb'


# name of feature-class within geodatabase
feature_class_name = 'BasinATLAS_v10_lev12'

# create complete path to Feature-Class
feature_class_path = f'{gdb_path}\\{feature_class_name}'

# use dask_geopandas to read in feature class of BasinATLAS_v10_lev12: 

start = time.time()
layers = fiona.listlayers(gdb_path)
layers
selected_layer = layers[11]

# read in layer 'BasinATLAS_v10_lev12'
#subbasins_lev12 = gpd.GeoDataFrame.from_file(gdb_path,layer= 'BasinATLAS_v10_lev12')
data_level12 = dask_geopandas.read_file(gdb_path,layer= 'BasinATLAS_v10_lev12', npartitions = 4)
end = time.time()

runtime = (end - start)
print(runtime)

# now put partitions together again: to obtain one large df:

start = time.time()
subbasins_lev12 = data_level12.compute()
end= time.time()
dauer = (end-start)
print(f'{dauer}sec')


1.535308837890625
185.4024896621704sec


# read in stations shapefile containing merged ID's:

In [6]:
# read in station data with added info about HYBAS_L12 and closest two HydroRiverID'S
stations = gpd.read_parquet(
    '../../output_data/merge_0km_dist_stations/unique_stations_aggregated_IDs_0km_dist.parquet')

In [7]:
wq_data = pd.read_csv('../../output_data/merge_0km_dist_stations/wq_data_aggregated_0km_dist_new_ID.csv')

C:\Users\bartusch\AppData\Local\Temp\ipykernel_18224\2017737745.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  wq_data = pd.read_csv('../../output_data/merge_0km_dist_stations/wq_data_aggregated_0km_dist_new_ID.csv')


In [8]:
wq_data['HYBAS_ID'] = wq_data['HYBAS_ID'].astype(float).astype(pd.Int64Dtype(), errors='ignore')
wq_data['obs_date'] = pd.to_datetime(wq_data['obs_date'])
wq_data.head(10)

,Unnamed: 0,dataset,obs_date,NO3N,NH4N,NO2N,TOC,DOC,TP,DIP,...,DOC_F,TP_F,DIP_F,NO2N_NO3N,DIN,Q,OPO4,HYBAS_ID,merged_origins,Site_id
0,0,GRQA,1979-10-31,NaN,0.020002,NaN,14.999997,NaN,0.020009,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,NaN,"('GRQA', '100001')"
1,1,GRQA,1979-12-12,NaN,0.040004,NaN,NaN,NaN,0.010004,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,NaN,"('GRQA', '100001')"
2,2,GRQA,1980-01-29,NaN,0.020002,NaN,26.999995,NaN,0.020009,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,NaN,"('GRQA', '100001')"
3,3,GRQA,1980-02-26,NaN,0.040004,NaN,NaN,25.999996,0.020009,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,NaN,"('GRQA', '100001')"
4,4,GRQA,1980-04-30,NaN,0.040004,NaN,6.999999,NaN,0.020009,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,NaN,"('GRQA', '100001')"
5,5,GRQA,1980-05-20,NaN,0.030003,NaN,NaN,590.000030,0.010004,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,NaN,"('GRQA', '100001')"
6,6,GRQA,1980-06-25,NaN,0.119998,NaN,4.999999,NaN,0.020009,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,NaN,"('GRQA', '100001')"
7,7,GRQA,1980-09-03,NaN,0.010001,NaN,NaN,5.500005,0.030013,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,NaN,"('GRQA', '100001')"
8,8,GRQA,1980-09-23,NaN,0.010001,NaN,9.999998,NaN,0.020009,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,NaN,"('GRQA', '100001')"
9,9,GRQA,1981-01-13,NaN,0.040004,NaN,6.599996,NaN,0.004987,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7120992500,NaN,"('GRQA', '100001')"


### Exclude stations which can not be assigned to any HydroBASIN because they are in marine waters, delta regions or on small islands: 

In [9]:
wq_data_hybas = wq_data[wq_data['HYBAS_ID'].notna()]

In [10]:
wq_data_hybas['Site_id'].nunique()


71037

### Add some information of closest and second closest  river to water quality data: 

In [11]:
wq_hybas_info = wq_data_hybas.merge(stations[['1stNEAR_DIST_m', '1stHYRIV_ID',
       '1stORD_STRA','2ndNEAR_DIST_m',
       'flag_samerivID', '2ndHYRIV_ID', '2ndORD_STRA', 'merged_dataset', 'merged_ids', 'Station_ID']], how = 'left', left_on = 'Site_id', right_on = 'Station_ID').drop('Station_ID', axis = 1)

### aggregate wq_data by subbasin and observation_date: 

### input water quality data are grouped by HydroBasin and observation date and subsequently aggregated: mean, median...

In [12]:
output = []
input_data = wq_hybas_info.copy()
compounds = ['NO3N', 'NH4N', 'NO2N', 'TOC', 'DOC', 'TP', 'DIP', 'NO2N_NO3N', 'DIN', 'OPO4']

for comp in compounds: 
    print(comp)
    input_data[comp] = input_data[comp].astype(float).values
    input_data['year'] = input_data['obs_date'].dt.year
    comp_data = input_data[['HYBAS_ID', 'Site_id', comp, 'obs_date', 'year']].copy().dropna()
    comp_data[f'{comp}_log'] = np.log(comp_data[comp])
    group = comp_data.groupby(['HYBAS_ID', 'obs_date'])
    # stats
    median = group[[comp]].median().rename(columns = {comp:f'{comp}_median'})
    q25 = group[[comp]].quantile(0.25).rename(columns={comp:f'{comp}_25'})
    q75 = group[[comp]].quantile(0.75).rename(columns={comp:f'{comp}_75'})
    mean = group[[comp]].mean().rename(columns = {comp:f'{comp}_mean'})
    sd = group[[comp]].std().rename(columns = {comp:f'{comp}_sd'})
    min = group[[comp]].min().rename(columns = {comp:f'{comp}_min'})
    max = group[[comp]].max().rename(columns = {comp:f'{comp}_max'})
    n = group[[comp]].count().rename(columns = {comp:f'{comp}_n'})

    # log mean and log_sd:
    log_mean =  group[[f'{comp}_log']].mean().rename(columns = {f'{comp}_log':f'{comp}_logmean'})
    log_sd =group[[f'{comp}_log']].std().rename(columns = {f'{comp}_log':f'{comp}_logsd'})
    #start = group[['obs_date']].min().rename(columns = {'obs_date':f'{comp}_start'})
    #end = group[['obs_date']].max().rename(columns = {'obs_date':f'{comp}_end'})
    #nyears = group[['year']].nunique().rename(columns = {'year':f'{comp}_nyears'})
    n_station = group[['Site_id']].nunique().rename(columns = {'Site_id':f'n_stations_{comp}'})

    stats = pd.concat([median, q25, q75, mean, sd, min, max, n, log_mean, log_sd,  n_station], axis = 1)
    output.append(stats.copy())
 # get info about strahler order of available stations:   
group_all = input_data.groupby(['HYBAS_ID', 'obs_date'])

basins_data_daily = pd.concat(output,axis=1).sort_index()     
    


NO3N
NH4N
NO2N
TOC
DOC
TP
DIP
NO2N_NO3N
DIN
OPO4


In [13]:
basins_data_daily = basins_data_daily.sort_index().reset_index()

In [14]:
basins_data_daily['HYBAS_ID'] =basins_data_daily['HYBAS_ID'].astype(float).astype(pd.Int64Dtype())#, errors='ignore')

### Merge  basins_data_daily and HYDROBASIN shapefile:

In [11]:
subbasins_lev12['HYBAS_ID'] = subbasins_lev12['HYBAS_ID'].astype(float).astype(pd.Int64Dtype())#, errors='ignore')
subbasins_lev12.head(10)

,HYBAS_ID,NEXT_DOWN,NEXT_SINK,MAIN_BAS,DIST_SINK,DIST_MAIN,SUB_AREA,UP_AREA,PFAF_ID,ENDO,...,hft_ix_s09,hft_ix_u09,gad_id_smj,gdp_ud_sav,gdp_ud_ssu,gdp_ud_usu,hdi_ix_sav,Shape_Length,Shape_Area,geometry
0,1120000010,0.000000e+00,1.120000e+09,1.120000e+09,0.0,0.0,11.0,11.0,1.110110e+11,0,...,460,477,69,10613,1.726720e+09,1.726720e+09,691,0.168222,0.001023,"MULTIPOLYGON (((32.50000 29.94583, 32.50000 29..."
1,1120000020,0.000000e+00,1.120000e+09,1.120000e+09,0.0,0.0,137.0,416.8,1.110110e+11,0,...,233,144,69,10613,8.512022e+08,8.545347e+08,691,0.484738,0.012790,"MULTIPOLYGON (((32.36250 29.97083, 32.36193 29..."
2,1121694330,1.120000e+09,1.120000e+09,1.120000e+09,19.5,19.5,135.1,280.0,1.110110e+11,0,...,120,101,69,10613,3.274106e+06,3.332646e+06,691,0.534764,0.012620,"MULTIPOLYGON (((32.36250 29.96667, 32.35591 29..."
3,1121693980,1.121694e+09,1.120000e+09,1.120000e+09,35.3,35.3,144.9,144.9,1.110110e+11,0,...,83,83,69,10613,5.854053e+04,5.854053e+04,691,0.534832,0.013540,"MULTIPOLYGON (((32.25833 29.99167, 32.24341 29..."
4,1120000030,0.000000e+00,1.120000e+09,1.120000e+09,0.0,0.0,186.8,186.9,1.110110e+11,0,...,210,355,69,10613,1.196149e+08,1.196149e+08,691,0.799328,0.017419,"MULTIPOLYGON (((32.40000 29.73750, 32.39583 29..."
5,1120000040,0.000000e+00,1.120000e+09,1.120000e+09,0.0,0.0,235.6,235.6,1.110110e+11,0,...,64,64,69,10613,0.000000e+00,0.000000e+00,691,0.790074,0.021979,"MULTIPOLYGON (((32.30833 29.99583, 32.31076 29..."
6,1120000050,0.000000e+00,1.120000e+09,1.120000e+09,0.0,0.0,8.3,8.3,1.110110e+11,0,...,308,303,69,10613,4.232450e+05,4.232450e+05,691,0.141571,0.000769,"MULTIPOLYGON (((32.37500 29.69583, 32.37465 29..."
7,1120000060,0.000000e+00,1.120000e+09,1.120000e+09,0.0,0.0,161.5,328.6,1.110110e+11,0,...,95,79,69,10613,0.000000e+00,0.000000e+00,691,0.715052,0.015053,"MULTIPOLYGON (((32.35417 29.68750, 32.35174 29..."
8,1121696210,1.120000e+09,1.120000e+09,1.120000e+09,22.6,22.6,167.1,167.1,1.110110e+11,0,...,64,64,69,10613,0.000000e+00,0.000000e+00,691,0.560446,0.015591,"MULTIPOLYGON (((32.27083 29.85000, 32.25508 29..."
9,1120000070,0.000000e+00,1.120000e+09,1.120000e+09,0.0,0.0,2.6,2.6,1.110110e+11,0,...,290,292,69,10613,0.000000e+00,0.000000e+00,691,0.065645,0.000241,"MULTIPOLYGON (((32.36667 29.67917, 32.36424 29..."


In [12]:
basins_data_geo = basins_data_daily.merge(subbasins_lev12[['HYBAS_ID','Shape_Length', 'Shape_Area', 'geometry']], how = 'left', left_on = 'HYBAS_ID', right_on = 'HYBAS_ID')

### save data:

In [14]:
# first as csv file wq data with HYBAS_ID but without HYBAS attributes and geographical information:
basins_data_daily.to_csv('../../output_data/aggregate_stations_by_subbasins/daily_wq_data_by_subbasin_0km_merged.csv')

In [15]:
ts_start = time.time()
pickle.dump(basins_data_geo,open('../../output_data/aggregate_stations_by_subbasins/daily_wq_data_by_subbasin_0km_merged.pkl','wb'))
print("Elapsed %.2f seconds" % ((time.time() - ts_start)))



Elapsed 171.21 seconds
